# Notebook 10 — Cross-domain validation of the preprocessing findings: Flipkart, IMDB and Yelp (text only), 3 seeds

**Purpose (and its limits).** On Amazon (Notebooks 1 and 9) conventional cleaning hurt, and ECCM added nothing over minimally processed
text. This notebook checks whether those two **preprocessing** findings hold on the base paper's other text datasets, with **the same
code** (Notebook 1's ECCM, cleaning, model and training, unchanged). It is a cross-domain validation, **not** a new method and **not**
multimodal. Its tests were written before Notebook 9's results were seen.

**Runs** — 3 datasets × 3 runs × 3 seeds = **27 fine-tunings** (RoBERTa-base, max 128 tokens, 2 epochs, batch 32, as Notebook 1;
best epoch chosen on validation macro-F1):
- **R1 — conventional cleaning:** lowercase; URLs, emojis, digits, punctuation and stop-words removed (the list includes *not*, *no*, *never*).
- **R2 — minimally processed text:** the review is only split into words and punctuation marks by a regular expression; nothing is
  removed or changed (Notebook 1's "raw" input).
- **R3 — ECCM cue-based processing without domain anchoring:** R2's input plus ECCM's five cue flags per word (negation, negation scope,
  intensifier, contrast, informal) with learnable cue vectors and intensity weights. ECCM's domain anchoring (PMI) needs product
  categories; these datasets have none, so the PMI term is 0 for every word.

**Not tested here:** Group-DRO (no category groups); photos (Yelp's photos belong to businesses, not to reviews; Flipkart and IMDB have none).

**Data (sizes follow the base paper's table where the source allows)**
| Dataset | Source | Labels | Cleaning | Train / valid / test |
|---|---|---|---|---|
| Flipkart | HF `KayEe/flipkart_sentiment_analysis` (your Drive copy if found) | given labels (positive / neutral / negative); provenance not documented | the official split **leaks** (2,403 texts in both) → pooled; exact duplicates removed; texts with conflicting labels removed (861); ≥ 3 words | 50,000 sampled → 40,000 / 5,000 / 5,000 (stratified) |
| IMDB | HF `stanfordnlp/imdb` (Drive copy if found) | negative / positive (2 classes) | duplicates removed; test texts also in train removed; ≥ 3 words | official train → 90 % / 10 % (stratified); official test |
| Yelp | HF `Yelp/yelp_review_full` (downloaded once into Drive) | stars 1–2 negative, 3 neutral, 4–5 positive (as Amazon) | duplicates removed; ≥ 3 words | 90,000 from the official train → 80,000 / 10,000; 10,000 from the official test |
Caveats: Flipkart here is mostly English with some Hinglish, **not** the paper's code-mixed set; Yelp keeps the source's balance
(40 / 20 / 40 after mapping), not the paper's 55 / 15 / 30; IMDB has no neutral class. Numbers are not comparable with the paper's.

**Checks before any training (Section 3b).** For every dataset: source file found and its SHA-256 compared with the file profiled before
this notebook was written (a different hash is **flagged** in red text and recorded, not silently accepted); schema (required columns, text
type, allowed label values); the label mapping, printed with counts; train / valid / test **disjoint**; every class present in every split
(at least 20 reviews); a fingerprint of the final split. Training starts only if all three datasets pass.
**Resume checks (Section 5).** A finished run is skipped only if its saved settings equal this notebook's, its dataset fingerprint equals
the current split's, and its prediction file loads with the right shape and valid probabilities. Different settings or data → the notebook
**stops** (nothing is overwritten). A run whose prediction file is missing or broken is renamed "_incomplete_…" and trained again.

**Tests (fixed in advance)** — per dataset: **T1 = R2 − R1** (does conventional cleaning hurt?) and **T2 = R3 − R2** (does ECCM's cue-based
processing help?), test macro-F1. **Holm over the 6 tests.** Interval and p-value: 10,000 bootstrap draws resampling **individual test
reviews**, together with seeds. **These datasets have no product or business identifiers, so reviews of the same product or business
cannot be resampled together; if such reviews are correlated, the intervals are too narrow.** Verdicts as before: HELPS / HURTS =
|effect| ≥ 1 pp and Holm p < .05; negligible = 95 % CI inside ±1 pp; otherwise no detectable effect. Secondary (Holm over 6): the same
comparisons on test reviews that contain a negation word (ECCM's main target). Descriptive: accuracy, ECE.

**Runtime: GPU (L4), High-RAM.** About 3.5 h on an L4 (from Notebook 9's measured L4 speed, about 0.135 s per training step); a
projection is printed after every run. Order: Flipkart → IMDB → Yelp. Each run is saved when it finishes; after a disconnect,
**Run all** continues. Writes only to `emofar_results/text_v10/`.

In [1]:
import os, sys, time, json, gc, glob, hashlib
IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("EMOFAR_TEST") == "1"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import torch, warnings
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", category=UserWarning)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU")
if IN_COLAB and DEVICE != "cuda":
    raise RuntimeError("Please switch to a GPU runtime (Runtime -> Change runtime type -> L4 GPU).")

Mounted at /content/drive
Device: cuda | NVIDIA L4


## 1. Settings (training settings identical to Notebook 1)

In [2]:
DATA_ROOT = "/content/drive/MyDrive/datasets"
OUT_DIR = "/content/drive/MyDrive/emofar_results/text_v10"
MODEL_NAME = "roberta-base"
MAX_LEN, EPOCHS, BATCH_SIZE = 128, 2, 32
LR_ENCODER, LR_NEW = 2e-5, 1e-4
SEEDS = [42, 43, 44]
GDRO_ETA, IMG_DROPOUT, USE_IMAGES = 0.01, 0.0, False     # names read by Notebook 1's code; not used here (text only, no Group-DRO)
SIZES = {"Flipkart": dict(total=50000, valid=5000, test=5000), "IMDB": dict(valid_frac=0.10),
         "Yelp": dict(train_pool=90000, valid=10000, test=10000)}
DATASETS = ["Flipkart", "IMDB", "Yelp"]
N_BOOT, MARGIN_PP, ALPHA = 10000, 1.0, 0.05
if TEST_MODE:
    DATA_ROOT, OUT_DIR = os.environ["EMOFAR_DATA"], os.environ["EMOFAR_OUT"]
    MODEL_NAME, MAX_LEN, BATCH_SIZE, EPOCHS = "distilroberta-base", 64, 16, 1
    SIZES = {"Flipkart": dict(total=300, valid=60, test=60), "IMDB": dict(valid_frac=0.10, max_train=250, max_test=80),
             "Yelp": dict(train_pool=300, valid=60, test=80)}
    N_BOOT = 500
os.makedirs(OUT_DIR, exist_ok=True)
CONFIG = dict(MODEL_NAME=MODEL_NAME, MAX_LEN=MAX_LEN, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR_ENCODER=LR_ENCODER, LR_NEW=LR_NEW, SIZES=SIZES)
import transformers, platform
print({"torch": torch.__version__, "transformers": transformers.__version__})
json.dump({"config": CONFIG, "seeds": SEEDS, "torch": torch.__version__, "transformers": transformers.__version__,
           "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"},
          open(f"{OUT_DIR}/run_info_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}.json", "w"), indent=2)

{'torch': '2.14.1+cu130', 'transformers': '5.22.1'}


## 2. ECCM, conventional cleaning and encoding (Notebook 1, unchanged)

In [3]:
import re, random
import numpy as np, pandas as pd

NEGATIONS = {"not", "no", "never", "nothing", "nobody", "none", "neither", "nor", "cannot", "without",
             "hardly", "barely", "scarcely", "nowhere", "cant", "dont", "doesnt", "didnt", "isnt", "wasnt",
             "wont", "wouldnt", "shouldnt", "couldnt", "arent", "werent", "havent", "hasnt", "hadnt", "aint"}
INTENSIFIERS = {"very", "really", "so", "too", "extremely", "super", "totally", "absolutely", "highly",
                "completely", "incredibly", "quite", "truly", "most", "much", "way", "seriously",
                "terribly", "awfully", "insanely", "utterly", "especially", "particularly", "even"}
CONTRAST = {"but", "however", "although", "though", "yet", "except", "whereas", "nevertheless", "unfortunately"}
SCOPE_BREAK = set(".,!?;:") | CONTRAST
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F2FF]")
WORD_RE = re.compile(r"\w+(?:'\w+)?|[^\w\s]")
FLAG_NAMES = ["NEG", "SCOPE", "INT", "CON", "INF"]

def eccm_words(text):
    """Split text into words and compute the 5 cue flags for each word."""
    words, flags = [], []
    for w in WORD_RE.findall(text):
        informal = bool(EMOJI_RE.search(w)) or bool(re.search(r"(\w)\1{2,}", w)) \
                   or (w.isupper() and len(w) >= 3 and w.isalpha()) or w == "!"
        w_norm = re.sub(r"(\w)\1{2,}", r"\1\1", w)                    # sooooo -> soo
        low = w_norm.lower().replace("'", "")
        f = [0, 0, 0, 0, 0]
        f[0] = int(low in NEGATIONS or w_norm.lower().endswith("n't"))
        f[2] = int(low in INTENSIFIERS)
        f[3] = int(low in CONTRAST)
        f[4] = int(informal)
        words.append(w_norm); flags.append(f)
    scope_left = 0                                                     # negation scope: next <=3 words
    for i, w in enumerate(words):
        if flags[i][0]:
            scope_left = 3; continue
        if w.lower() in SCOPE_BREAK:
            scope_left = 0; continue
        if scope_left > 0:
            flags[i][1] = 1; scope_left -= 1
    return words, flags

w, f = eccm_words("Not very good, but the colour is sooooo NICE 😍!!")
for word, fl in zip(w, f):
    print(f"{word:8s}", [n for n, v in zip(FLAG_NAMES, fl) if v])

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print("Is 'not' a stop-word?", "not" in ENGLISH_STOP_WORDS, "| 'never'?", "never" in ENGLISH_STOP_WORDS)

def conventional_clean(text):
    t = re.sub(r"https?://\S+", " ", text.lower())
    t = re.sub(r"[^a-z\s]", " ", t)                  # drops emojis, digits, punctuation, apostrophes
    return " ".join(w for w in t.split() if w not in ENGLISH_STOP_WORDS)

print(conventional_clean("Not very good, but the colour is sooooo NICE 😍!!"))

from collections import Counter, defaultdict

ANCHOR_SKIP = set(ENGLISH_STOP_WORDS) - NEGATIONS     # filler words are not domain words

def pmi(word, cat, min_count=10):
    c_all = word_all.get(word, 0)
    if c_all < min_count or not word.isalpha() or len(word) < 3 or word in ANCHOR_SKIP:
        return 0.0
    p_w_given_c = word_cat[cat].get(word, 0) / max(cat_total[cat], 1)
    p_w = c_all / N_TRAIN
    if p_w_given_c == 0:
        return 0.0
    return float(np.clip(np.log(p_w_given_c / p_w) / 3.0, 0.0, 1.0))

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)

def encode(frame, mode):
    """mode: 'conventional' | 'raw' | 'eccm'. Returns a dict of tensors."""
    ids, masks, flag_t, pmi_t = [], [], [], []
    for text, (words, flags), cat in zip(frame.text, frame.eccm, frame.category):
        if mode == "conventional":
            words = conventional_clean(text).split() or ["empty"]
            flags = [[0] * 5 for _ in words]
        elif mode == "raw":
            words = WORD_RE.findall(text); flags = [[0] * 5 for _ in words]
        enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=MAX_LEN,
                        padding="max_length")
        wid = enc.word_ids()
        fl = [[0] * 5 if i is None else flags[i] for i in wid]
        pm = [0.0 if (i is None or mode != "eccm") else pmi(words[i].lower(), cat) for i in wid]
        ids.append(enc["input_ids"]); masks.append(enc["attention_mask"]); flag_t.append(fl); pmi_t.append(pm)
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks),
            "flags": torch.tensor(flag_t, dtype=torch.float), "pmi": torch.tensor(pmi_t, dtype=torch.float),
            "label": torch.tensor(frame.label.values), "group": torch.tensor([CATEGORIES.index(c) for c in frame.category]),
            "row": torch.tensor(frame.index.values)}

Not      ['NEG']
very     ['SCOPE', 'INT']
good     ['SCOPE']
,        []
but      ['CON']
the      []
colour   []
is       []
soo      ['INF']
NICE     ['INF']
😍        ['INF']
!        ['INF']
!        ['INF']
Is 'not' a stop-word? True | 'never'? True
good colour sooooo nice


## 3. Data: load, check, clean, de-duplicate, split (built once; the same split for every run)

In [4]:
from huggingface_hub import hf_hub_download

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_or_download(folder, patterns, repo, filename):
    """Use the copy in Drive if one matches; otherwise download from Hugging Face into Drive (once)."""
    for p in patterns:
        hits = sorted(glob.glob(f"{DATA_ROOT}/{folder}*/**/{p}", recursive=True))
        if hits:
            return hits[0]
    return hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=f"{DATA_ROOT}/{folder}_hf")

EXPECTED_SHA256 = {   # the Hugging Face files profiled before this notebook was written
    "Flipkart": {"train": "4f53e0a45bd5b61fcb7cd43884c1561122ce17d4aea9334b551055a99ffa9943", "test": "9cb489adce35ba808b3d898a7907b02329a652ffaae08aff7f7c11706b1ab801"},
    "IMDB": {"train": "db47d16b5c297cc0dd625e519c81319c24c9149e70e8496de5475f6fa928342c", "test": "b52e26e2f872d282ffac460bf9770b25ac6f102cda0e6ca7158df98c94e8b3da"},
    "Yelp": {"train": "da3ca3dcc52b1e2e44195b53668610d645a59010a985dd45aa12e1109cd44601", "test": "bf06d5969bff93ecd4a6d4b330643761ce108b42e9b8a894cf82c9df02a08540"}}
EXPECTED_SCHEMA = {"Flipkart": {"text": "input", "label": "output", "values": {"positive", "negative", "neutral"}},
                   "IMDB": {"text": "text", "label": "label", "values": {0, 1}},
                   "Yelp": {"text": "text", "label": "label", "values": {0, 1, 2, 3, 4}}}
CLASS_NAMES = {"Flipkart": ["negative", "neutral", "positive"], "IMDB": ["negative", "positive"], "Yelp": ["negative", "neutral", "positive"]}
MIN_PER_CLASS = 1 if TEST_MODE else 20

def norm_key(t):
    return hashlib.md5(re.sub(r"\s+", " ", str(t).lower()).strip().encode()).hexdigest()

def star_to_label(stars):
    return 0 if stars <= 2 else (1 if stars == 3 else 2)

def fingerprint(d):
    return hashlib.md5("|".join(f"{k}:{s}:{l}" for k, s, l in sorted(zip(d.key, d.split, d.label))).encode()).hexdigest()

def check_raw(ds, sp, frame, path, report):
    """Raw-file checks: required columns, label values, text type, file hash. Stops on a schema problem; flags a different hash."""
    sch = EXPECTED_SCHEMA[ds]; tc, lc = sch["text"], sch["label"]
    missing = [c for c in (tc, lc) if c not in frame.columns]
    if missing:
        raise RuntimeError(f"{ds} {sp}: required columns {missing} not found in {path}; the file has {list(frame.columns)}")
    if frame[lc].isna().any():
        raise RuntimeError(f"{ds} {sp}: {int(frame[lc].isna().sum())} rows without a label")
    bad = set(frame[lc].unique().tolist()) - sch["values"]
    if bad:
        raise RuntimeError(f"{ds} {sp}: unexpected label values {sorted(map(str, bad))[:10]}")
    is_text = frame[tc].map(lambda x: isinstance(x, str) and len(x.strip()) > 0)
    h, exp = sha256(path), EXPECTED_SHA256[ds][sp]
    report["sources"][sp] = {"file": os.path.basename(path), "sha256": h, "expected_sha256": exp, "hash_matches": h == exp,
                             "rows": int(len(frame)), "columns": [str(c) for c in frame.columns], "rows_without_text": int((~is_text).sum()),
                             "raw_label_counts": {str(k): int(v) for k, v in frame[lc].value_counts().sort_index().items()}}
    if h != exp:
        print(f"  \033[91m!! UNEXPECTED FILE HASH: {ds} {sp} ({os.path.basename(path)}) has SHA-256 {h[:16]}..., expected {exp[:16]}...\n"
              f"     Training continues; this is recorded in validation_report.json and must be reported with the results.\033[0m")
    else:
        print(f"  {ds} {sp}: {len(frame):,} rows; columns and label values as expected; file hash matches the profiled file")
    return frame[is_text]

def build_frame(ds):
    rs = 42; report = {"dataset": ds, "sources": {}}
    sch = EXPECTED_SCHEMA[ds]
    if ds == "Flipkart":
        parts = []
        for sp in ["train", "test"]:
            p = find_or_download("Flipkart", [f"{sp}.json"], "KayEe/flipkart_sentiment_analysis", f"{sp}.json")
            parts.append(check_raw(ds, sp, pd.DataFrame(json.load(open(p))), p, report)[["input", "output"]])
        d = pd.concat(parts).rename(columns={"input": "text"}); d["key"] = d.text.map(norm_key)
        mapping = {"negative": 0, "neutral": 1, "positive": 2}; d["label"] = d.output.map(mapping)
        conflict = d.groupby("key").label.nunique(); bad = set(conflict[conflict > 1].index)
        n0 = len(d); d = d[~d.key.isin(bad)]; n1 = len(d); d = d.drop_duplicates("key"); n2 = len(d)
        d = d[d.text.str.split().str.len() >= 3]
        report["cleaning"] = {"pooled": n0, "conflicting_texts_removed": len(bad), "after_conflicts": n1, "unique": n2, "at_least_3_words": len(d)}
        print(f"  pooled {n0:,} -> {n1:,} after removing {len(bad):,} conflicting texts -> {n2:,} unique -> {len(d):,} with >= 3 words")
        z = SIZES["Flipkart"]; d = d.sample(n=min(z["total"], len(d)), random_state=rs).reset_index(drop=True)
        split = np.array(["train"] * len(d), dtype=object); rng = np.random.RandomState(rs)
        for lab in range(3):
            idx = rng.permutation(np.where(d.label.values == lab)[0]); nv = round(z["valid"] * len(idx) / len(d)); nt = round(z["test"] * len(idx) / len(d))
            split[idx[:nv]] = "valid"; split[idx[nv:nv + nt]] = "test"
        d["split"] = split; n_cls = 3; report["label_mapping"] = {"source -> class": mapping}
        raw_vs_mapped = pd.crosstab(d.output, d.label)
    elif ds == "IMDB":
        fr = {}
        for sp in ["train", "test"]:
            p = find_or_download("IMDB", [f"{sp}-*.parquet", f"{sp}.parquet"], "stanfordnlp/imdb", f"plain_text/{sp}-00000-of-00001.parquet")
            fr[sp] = check_raw(ds, sp, pd.read_parquet(p), p, report)[["text", "label"]]; fr[sp]["key"] = fr[sp].text.map(norm_key)
        tr = fr["train"].drop_duplicates("key"); te = fr["test"].drop_duplicates("key"); n_te0 = len(te); te = te[~te.key.isin(set(tr.key))]
        tr = tr[tr.text.str.split().str.len() >= 3]; te = te[te.text.str.split().str.len() >= 3]
        if "max_train" in SIZES["IMDB"]:
            tr = tr.sample(n=SIZES["IMDB"]["max_train"], random_state=rs); te = te.sample(n=SIZES["IMDB"]["max_test"], random_state=rs)
        rng = np.random.RandomState(rs); tr = tr.reset_index(drop=True); split = np.array(["train"] * len(tr), dtype=object)
        for lab in range(2):
            idx = rng.permutation(np.where(tr.label.values == lab)[0]); split[idx[:round(SIZES["IMDB"]["valid_frac"] * len(idx))]] = "valid"
        tr["split"] = split; te = te.assign(split="test")
        d = pd.concat([tr, te]).reset_index(drop=True); n_cls = 2
        report["cleaning"] = {"train_unique": int(len(fr["train"].key.unique())), "test_unique": n_te0, "test_removed_seen_in_train": int(n_te0 - len(te))}
        report["label_mapping"] = {"source -> class": {"0 (negative)": 0, "1 (positive)": 1}}
        raw_vs_mapped = pd.crosstab(d.label, d.label)
        print(f"  after de-duplication and removing test texts seen in train: train+valid {len(tr):,}, test {len(te):,}")
    else:
        fr = {}
        for sp in ["train", "test"]:
            p = find_or_download("Yelp", [f"{sp}-*.parquet"], "Yelp/yelp_review_full", f"yelp_review_full/{sp}-00000-of-00001.parquet")
            fr[sp] = check_raw(ds, sp, pd.read_parquet(p), p, report)[["text", "label"]]
        z = SIZES["Yelp"]                                     # draw a random pool first (650,000 rows are not all needed)
        fr["train"] = fr["train"].sample(n=min(len(fr["train"]), int(z["train_pool"] * 1.2)), random_state=rs)
        for sp in fr:
            fr[sp] = fr[sp][fr[sp].text.str.split().str.len() >= 3]
            fr[sp]["key"] = fr[sp].text.map(norm_key); fr[sp] = fr[sp].drop_duplicates("key")
            fr[sp]["stars"] = fr[sp].label + 1; fr[sp]["label"] = fr[sp].stars.map(star_to_label)
        tr = fr["train"].sample(n=z["train_pool"], random_state=rs).reset_index(drop=True)
        te = fr["test"][~fr["test"].key.isin(set(tr.key))].sample(n=z["test"], random_state=rs)
        rng = np.random.RandomState(rs); split = np.array(["train"] * len(tr), dtype=object)
        for lab in range(3):
            idx = rng.permutation(np.where(tr.label.values == lab)[0]); split[idx[:round(z["valid"] * len(idx) / len(tr))]] = "valid"
        tr["split"] = split; te = te.assign(split="test"); d = pd.concat([tr, te]).reset_index(drop=True); n_cls = 3
        report["label_mapping"] = {"source label 0-4 = stars 1-5 -> class": {"1-2 stars": 0, "3 stars": 1, "4-5 stars": 2}}
        raw_vs_mapped = pd.crosstab(d.stars, d.label); del fr
    print(f"  label mapping (rows: source label, columns: class {list(range(n_cls))} = {CLASS_NAMES[ds]}):")
    print("    " + raw_vs_mapped.to_string().replace("\n", "\n    "))
    report["mapped_counts"] = {str(k): {str(c): int(v) for c, v in row.items()} for k, row in raw_vs_mapped.iterrows()}
    d = d[["text", "label", "key", "split"]].reset_index(drop=True); d["category"] = ds
    return d, n_cls, report

def validate_frame(ds, d, n_cls, report):
    """Split-level checks. Stops if any fails."""
    keys = {sp: set(d.key[d.split == sp]) for sp in ["train", "valid", "test"]}
    overlaps = {f"{a}&{b}": len(keys[a] & keys[b]) for a, b in [("train", "valid"), ("train", "test"), ("valid", "test")]}
    counts = pd.crosstab(d.split, d.label).reindex(index=["train", "valid", "test"], columns=range(n_cls), fill_value=0)
    problems = []
    if any(overlaps.values()): problems.append(f"splits share texts {overlaps}")
    if d.key.duplicated().any(): problems.append(f"{int(d.key.duplicated().sum())} duplicate texts")
    if not d.label.between(0, n_cls - 1).all(): problems.append("labels outside the class range")
    if (counts.values < MIN_PER_CLASS).any(): problems.append(f"a class has fewer than {MIN_PER_CLASS} reviews in some split:\n{counts}")
    if d.text.str.split().str.len().min() < 3: problems.append("texts shorter than 3 words remain")
    report.update({"split_overlaps": overlaps, "counts": {sp: {CLASS_NAMES[ds][c]: int(counts.loc[sp, c]) for c in range(n_cls)} for sp in counts.index},
                   "n_classes": n_cls, "fingerprint": report.get("fingerprint") or fingerprint(d), "checks_passed": not problems})
    if problems:
        raise RuntimeError(f"{ds}: data checks failed: " + "; ".join(problems))
    return report

## 3b. Check all three datasets before any training
Training starts only if every dataset passes. A saved split is re-checked against its recorded fingerprint before it is reused.

In [5]:
VALID = {}
for ds in DATASETS:
    os.makedirs(f"{OUT_DIR}/{ds}", exist_ok=True); fpath = f"{OUT_DIR}/{ds}/data.pkl"
    if os.path.exists(fpath):
        d, n_cls, report = pd.read_pickle(fpath)
        if fingerprint(d) != report["fingerprint"]:
            raise RuntimeError(f"{ds}: the saved split does not match its recorded fingerprint - stop and report")
        print(f"{ds}: saved split loaded; fingerprint {report['fingerprint'][:12]}... verified")
    else:
        print(f"{ds}: building the split")
        d, n_cls, report = build_frame(ds); report["fingerprint"] = fingerprint(d)
        validate_frame(ds, d, n_cls, report)
        pd.to_pickle((d, n_cls, report), fpath + ".tmp"); os.replace(fpath + ".tmp", fpath)
    validate_frame(ds, d, n_cls, report)
    json.dump(report, open(f"{OUT_DIR}/{ds}/validation_report.json", "w"), indent=2, default=str)
    VALID[ds] = report; del d
summary = pd.DataFrame([{"dataset": ds, "classes": r["n_classes"], **{f"n_{sp}": sum(r["counts"][sp].values()) for sp in ["train", "valid", "test"]},
                         "smallest class (any split)": min(min(v.values()) for v in r["counts"].values()),
                         "splits disjoint": not any(r["split_overlaps"].values()),
                         "source hashes match": all(s["hash_matches"] for s in r["sources"].values()), "fingerprint": r["fingerprint"][:12]}
                        for ds, r in VALID.items()])
print(summary.to_string(index=False))
HASH_FLAGS = [f"{ds} {sp}" for ds, r in VALID.items() for sp, s in r["sources"].items() if not s["hash_matches"]]
print("\nAll three datasets passed the checks; training may start." +
      (f" \033[91mUNEXPECTED FILE HASHES (recorded, must be reported): {HASH_FLAGS}\033[0m" if HASH_FLAGS else " All source file hashes match the profiled files."))

Flipkart: building the split
  Flipkart train: 126,265 rows; columns and label values as expected; file hash matches the profiled file
  Flipkart test: 54,113 rows; columns and label values as expected; file hash matches the profiled file
  pooled 180,378 -> 111,178 after removing 861 conflicting texts -> 84,444 unique -> 79,167 with >= 3 words
  label mapping (rows: source label, columns: class [0, 1, 2] = ['negative', 'neutral', 'positive']):
    label         0     1      2
    output                      
    negative  10063     0      0
    neutral       0  2460      0
    positive      0     0  37477
IMDB: building the split
  IMDB train: 25,000 rows; columns and label values as expected; file hash matches the profiled file
  IMDB test: 25,000 rows; columns and label values as expected; file hash matches the profiled file
  after de-duplication and removing test texts seen in train: train+valid 24,904, test 24,678
  label mapping (rows: source label, columns: class [0, 1] = ['neg

## 4. Model and training (Notebook 1, unchanged; IMDB uses a 2-class output layer)

In [6]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class EmoFarModel(nn.Module):
    def __init__(self, use_eccm=False, use_image=False, img_dim=768):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        H = self.encoder.config.hidden_size
        self.use_eccm, self.use_image = use_eccm, use_image
        if use_eccm:
            self.cue_emb = nn.Parameter(torch.randn(5, H) * 0.02)      # E_k, one vector per cue type
            self.alpha = nn.Parameter(torch.tensor([0.1, 0.1]))          # α_INT, α_INF
            self.beta = nn.Parameter(torch.tensor(0.1))                  # β (domain anchoring)
        if use_image:
            self.img_proj = nn.Sequential(nn.Linear(img_dim, H), nn.LayerNorm(H))
            self.cross_attn = nn.MultiheadAttention(H, num_heads=8, batch_first=True)
            self.gate = nn.Linear(2 * H, 2)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(H, 3)

    def forward(self, input_ids, attention_mask, flags, pmi, img=None, has_img=None, return_gate=False):
        emb = self.encoder.embeddings.word_embeddings(input_ids)        # (B, L, H)
        if self.use_eccm:
            s = 1 + self.alpha[0] * flags[..., 2] + self.alpha[1] * flags[..., 4]
            a = 1 + self.beta * pmi
            emb = emb * (a * s).unsqueeze(-1) + flags @ self.cue_emb
        tok = self.encoder(inputs_embeds=emb, attention_mask=attention_mask).last_hidden_state
        h_text = tok[:, 0]                                               # <s> token summary
        gate_w = None
        if self.use_image:
            v = self.img_proj(img)                                       # (B, 50, H)
            attended, _ = self.cross_attn(tok, v, v)                     # text tokens attend to image tokens
            m = attention_mask.unsqueeze(-1).float()
            h_img = (attended * m).sum(1) / m.sum(1).clamp(min=1)
            logits_g = self.gate(torch.cat([h_text, h_img], -1))
            logits_g = logits_g.masked_fill(torch.stack([torch.zeros_like(has_img), ~has_img], -1), float("-inf"))
            gate_w = logits_g.softmax(-1)                                # (B, 2): [w_text, w_image]
            h = gate_w[:, :1] * h_text + gate_w[:, 1:] * h_img
        else:
            h = h_text
        out = self.classifier(self.dropout(h))
        return (out, gate_w) if return_gate else out

EmoFarModel3 = EmoFarModel
class EmoFarModel2(EmoFarModel3):
    """Notebook 1's model with a 2-class output layer (IMDB has no neutral class). Nothing else changes."""
    def __init__(self, use_eccm=False, use_image=False, img_dim=768):
        super().__init__(use_eccm=use_eccm, use_image=use_image, img_dim=img_dim)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, 2)

import time
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from transformers import get_linear_schedule_with_warmup

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def batches(data, bs, shuffle, seed=0):
    n = len(data["label"])
    order = np.random.RandomState(seed).permutation(n) if shuffle else np.arange(n)
    for i in range(0, n, bs):
        yield {k: v[order[i:i + bs]] for k, v in data.items()}

def model_inputs(b, use_image, drop_image_p=0.0, rng=None):
    x = {k: b[k].to(DEVICE) for k in ["input_ids", "attention_mask", "flags", "pmi"]}
    if use_image:
        rows = b["row"].numpy()
        has = HAS_IMG[rows].copy()
        if drop_image_p > 0:
            has &= rng.rand(len(has)) >= drop_image_p                   # modality dropout / missing-image test
        img = torch.tensor(IMG_FEATS[rows], dtype=torch.float).to(DEVICE)
        img = img * torch.tensor(has, dtype=torch.float, device=DEVICE)[:, None, None]
        x.update(img=img, has_img=torch.tensor(has, device=DEVICE))
    return x

def ece_score(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    ece, edges = 0.0, np.linspace(0, 1, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.any():
            ece += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return float(ece)

@torch.no_grad()
def predict(model, data, use_image, drop_image_p=0.0, seed=0):
    model.eval(); probs = []
    rng = np.random.RandomState(seed)
    for b in batches(data, 64, False):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            out = model(**model_inputs(b, use_image, drop_image_p, rng))
        probs.append(out.float().softmax(-1).cpu().numpy())
    return np.concatenate(probs)

def evaluate(probs, data):
    y, g, pred = data["label"].numpy(), data["group"].numpy(), probs.argmax(1)
    rows = data["row"].numpy()
    has_neg = np.array([any(f[0] for f in df.loc[r, "eccm"][1]) for r in rows])
    per_group = {CATEGORIES[k]: f1_score(y[g == k], pred[g == k], average="macro") for k in np.unique(g)}
    res = {"accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
           "worst_group_f1": min(per_group.values()), "fairness_gap": max(per_group.values()) - min(per_group.values()),
           "ece": ece_score(probs, y),
           "negation_f1": f1_score(y[has_neg], pred[has_neg], average="macro") if has_neg.any() else float("nan")}
    res.update({f"f1_{k}": v for k, v in per_group.items()})
    return res

def train_run(name, mode, use_eccm, loss_type, use_image, seed):
    set_seed(seed)
    data = encoded[mode]
    model = EmoFarModel(use_eccm=use_eccm, use_image=use_image).to(DEVICE)
    enc_params = list(model.encoder.parameters())
    enc_ids = {id(p) for p in enc_params}
    new_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": LR_ENCODER}, {"params": new_params, "lr": LR_NEW}], weight_decay=0.01)
    steps = EPOCHS * int(np.ceil(len(data["train"]["label"]) / BATCH_SIZE))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    q = torch.ones(len(CATEGORIES), device=DEVICE) / len(CATEGORIES)    # Group-DRO weights
    rng = np.random.RandomState(seed)
    best_f1, best_state, t0 = -1, None, time.time()
    for ep in range(EPOCHS):
        model.train()
        for b in batches(data["train"], BATCH_SIZE, True, seed + ep):
            y, g = b["label"].to(DEVICE), b["group"].to(DEVICE)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                logits = model(**model_inputs(b, use_image, IMG_DROPOUT if use_image else 0.0, rng))
            per_sample = F.cross_entropy(logits.float(), y, reduction="none")
            if loss_type == "gdro":
                group_loss = torch.stack([per_sample[g == k].mean() if (g == k).any() else per_sample.new_zeros(())
                                          for k in range(len(CATEGORIES))])
                q = q * torch.exp(GDRO_ETA * group_loss.detach()); q = q / q.sum()
                loss = (q * group_loss).sum()
            else:
                loss = per_sample.mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        val = evaluate(predict(model, data["valid"], use_image), data["valid"])
        print(f"  [{name}] epoch {ep + 1}/{EPOCHS}  valid macro-F1 = {val['macro_f1']:.4f}  ({time.time() - t0:.0f}s)")
        if val["macro_f1"] > best_f1:
            best_f1 = val["macro_f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model

TEXT_RUNS = {"R1": ("conventional", False, "ce"), "R2": ("raw", False, "ce"), "R3": ("eccm", True, "ce")}
RUN_NAMES = {"R1": "R1 conventional cleaning", "R2": "R2 minimally processed text", "R3": "R3 ECCM cues, no domain anchoring"}

def load_split(ds):
    d, n_cls, report = pd.read_pickle(f"{OUT_DIR}/{ds}/data.pkl")
    assert fingerprint(d) == report["fingerprint"] == VALID[ds]["fingerprint"], f"{ds}: split fingerprint mismatch"
    return d, n_cls, report

def prepare(ds):
    """Loads the checked split and sets the globals that Notebook 1's functions read."""
    global df, CATEGORIES, word_cat, word_all, cat_total, N_TRAIN, train_df, encoded, EmoFarModel, N_CLS
    df, N_CLS, _ = load_split(ds)
    CATEGORIES = [ds]; EmoFarModel = EmoFarModel2 if N_CLS == 2 else EmoFarModel3
    df["eccm"] = df.text.map(eccm_words)                              # Notebook 1's PMI tables (one domain -> PMI is 0)
    train_df = df[df.split == "train"]
    word_cat, word_all, cat_total = defaultdict(Counter), Counter(), Counter()
    for (words, _), cat in zip(train_df.eccm, train_df.category):
        for w_ in set(x.lower() for x in words):
            word_cat[cat][w_] += 1; word_all[w_] += 1
        cat_total[cat] += 1
    N_TRAIN = len(train_df)
    splits = {s: df[df.split == s] for s in ["train", "valid", "test"]}
    t1 = time.time(); encoded = {mode: {s: encode(fr, mode) for s, fr in splits.items()} for mode in ["conventional", "raw", "eccm"]}
    print(f"{ds}: {len(df):,} reviews, {N_CLS} classes, encoded in {time.time() - t1:.0f}s, median words {df.text.str.split().str.len().median():.0f}")
    neg_share = np.mean([any(f_[0] for f_ in e[1]) for e in df[df.split == "test"].eccm])
    print(f"  test reviews containing a negation word: {100 * neg_share:.1f}%")

## 5. Train R1–R3 × 3 seeds per dataset (each run saved when it finishes; a projection is printed)
A run is skipped only after its saved settings, dataset fingerprint and prediction file are verified.

In [7]:
def run_status(ds, rid, seed):
    """'done' | 'todo' | 'incomplete' (prediction file missing or broken) | 'conflict' (other settings or other data)."""
    tag = f"{OUT_DIR}/{ds}/{rid}_seed{seed}"; r = VALID[ds]
    if not os.path.exists(tag + "_log.json"):
        return "todo", []
    log = json.load(open(tag + "_log.json")); hard = []
    if log.get("config") != CONFIG: hard.append("its training settings differ from this notebook's")
    if log.get("data_fingerprint") != r["fingerprint"]: hard.append("it was trained on a different split (fingerprint differs)")
    if hard:
        return "conflict", hard
    soft = []
    try:
        z = np.load(tag + "_probs.npz")
        for sp in ["valid", "test"]:
            a = z[sp]; n = sum(r["counts"][sp].values())
            if a.shape != (n, r["n_classes"]): soft.append(f"{sp} predictions have shape {a.shape}, expected {(n, r['n_classes'])}")
            elif not (np.isfinite(a).all() and np.allclose(a.sum(1), 1, atol=1e-3)): soft.append(f"{sp} predictions are not valid probabilities")
    except Exception as e:
        soft.append(f"prediction file missing or unreadable ({type(e).__name__})")
    return ("done" if not soft else "incomplete"), soft

def run(ds, rid, seed):
    tag = f"{OUT_DIR}/{ds}/{rid}_seed{seed}"
    status, why = run_status(ds, rid, seed)
    if status == "done":
        return json.load(open(tag + "_log.json")), True
    if status == "conflict":
        raise RuntimeError(f"{tag}: {'; '.join(why)}. Nothing was overwritten. Move or rename {OUT_DIR}/{ds} to start this dataset afresh.")
    if status == "incomplete":
        stamp = time.strftime("%Y%m%d_%H%M%S")
        for suf in ["_log.json", "_probs.npz"]:
            if os.path.exists(tag + suf):
                os.replace(tag + suf, f"{tag}_incomplete_{stamp}{suf}")
        print(f"  {ds} {rid} seed {seed}: earlier run incomplete ({'; '.join(why)}) - kept as *_incomplete_{stamp}*, training again")
    mode, use_eccm, loss_type = TEXT_RUNS[rid]; t1 = time.time()
    model = train_run(f"{ds} {rid}", mode, use_eccm, loss_type, False, seed); secs = time.time() - t1
    pv, pt = predict(model, encoded[mode]["valid"], False), predict(model, encoded[mode]["test"], False)
    np.savez(tag + "_probs.tmp.npz", valid=pv.astype(np.float32), test=pt.astype(np.float32)); os.replace(tag + "_probs.tmp.npz", tag + "_probs.npz")
    log = {"dataset": ds, "run": rid, "run_name": RUN_NAMES[rid], "seed": seed, "train_seconds": round(secs, 1), "config": CONFIG,
           "data_fingerprint": VALID[ds]["fingerprint"], "valid": evaluate(pv, encoded[mode]["valid"]), "test": evaluate(pt, encoded[mode]["test"])}
    json.dump(log, open(tag + "_log.tmp.json", "w"), indent=2, default=float); os.replace(tag + "_log.tmp.json", tag + "_log.json")
    assert run_status(ds, rid, seed)[0] == "done", "a just-saved run failed its own check"
    del model; gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return log, False

T_START, SECS_PER_STEP = time.time(), []
STEPS = {ds: EPOCHS * int(np.ceil(sum(VALID[ds]["counts"]["train"].values()) / BATCH_SIZE)) for ds in DATASETS}
for ds in DATASETS:
    states = {(r, s): run_status(ds, r, s) for r in TEXT_RUNS for s in SEEDS}
    if any(st == "conflict" for st, _ in states.values()):
        bad = [f"{r} seed {s}: {'; '.join(w)}" for (r, s), (st, w) in states.items() if st == "conflict"]
        raise RuntimeError(f"{ds}: saved runs do not match this notebook - {bad}. Nothing was overwritten.")
    if all(st == "done" for st, _ in states.values()):
        print(f"{ds}: all 9 runs verified (settings, data fingerprint, prediction files) - skipped"); continue
    prepare(ds)
    for s in SEEDS:
        for rid in TEXT_RUNS:
            log, loaded = run(ds, rid, s)
            if not loaded:
                SECS_PER_STEP.append(log["train_seconds"] / STEPS[ds])
            left = sum(STEPS[d2] for d2 in DATASETS for r2 in TEXT_RUNS for s2 in SEEDS if run_status(d2, r2, s2)[0] != "done")
            proj = f" | about {np.mean(SECS_PER_STEP) * left / 60:.0f} min of training left" if SECS_PER_STEP else ""
            print(f"[{ds} {RUN_NAMES[rid]} seed {s}] {'verified, loaded' if loaded else 'trained in %.0fs' % log['train_seconds']} | valid macro-F1 "
                  f"{log['valid']['macro_f1']:.4f} | test macro-F1 {log['test']['macro_f1']:.4f} | elapsed {(time.time() - T_START) / 60:.0f} min{proj}", flush=True)
    del encoded; gc.collect()

Flipkart: 50,000 reviews, 3 classes, encoded in 75s, median words 8
  test reviews containing a negation word: 24.9%
  [Flipkart R1] epoch 1/2  valid macro-F1 = 0.6686  (159s)
  [Flipkart R1] epoch 2/2  valid macro-F1 = 0.6706  (319s)
[Flipkart R1 conventional cleaning seed 42] trained in 325s | valid macro-F1 0.6706 | test macro-F1 0.6701 | elapsed 7 min | about 168 min of training left
  [Flipkart R2] epoch 1/2  valid macro-F1 = 0.6795  (160s)
  [Flipkart R2] epoch 2/2  valid macro-F1 = 0.7078  (321s)
[Flipkart R2 minimally processed text seed 42] trained in 322s | valid macro-F1 0.7078 | test macro-F1 0.7228 | elapsed 12 min | about 162 min of training left
  [Flipkart R3] epoch 1/2  valid macro-F1 = 0.6943  (161s)
  [Flipkart R3] epoch 2/2  valid macro-F1 = 0.7019  (323s)
[Flipkart R3 ECCM cues, no domain anchoring seed 42] trained in 324s | valid macro-F1 0.7019 | test macro-F1 0.7262 | elapsed 18 min | about 157 min of training left
  [Flipkart R1] epoch 1/2  valid macro-F1 = 0.6

## 6. Statistics and results (reads only saved files)

In [8]:
def boot_f1(y, pred, mask=None, n_boot=N_BOOT, chunk=500):
    """Bootstrap draws of macro-F1, resampling test reviews; identical draws for every model (fixed seed)."""
    if mask is not None:
        y, pred = y[mask], pred[mask]
    n, k = len(y), int(max(y.max(), pred.max())) + 1
    onehot = np.zeros((n, k * k), dtype=np.float32); onehot[np.arange(n), y * k + pred] = 1
    rs, out = np.random.RandomState(2026), []
    for st in range(0, n_boot, chunk):
        w = np.stack([np.bincount(rs.randint(0, n, n), minlength=n) for _ in range(min(chunk, n_boot - st))]).astype(np.float32)
        cm = (w @ onehot).reshape(-1, k, k); tp = np.diagonal(cm, axis1=1, axis2=2); fp = cm.sum(1) - tp; fn = cm.sum(2) - tp
        den = 2 * tp + fp + fn; present = (cm.sum(2) > 0) | (cm.sum(1) > 0)
        f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0)
        out.append((f1 * present).sum(1) / present.sum(1))
    return np.concatenate(out)

def _p(b):
    return min(1.0, 2 * (min((b <= 0).sum(), (b >= 0).sum()) + 1) / (len(b) + 1))

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run_ = 0.0
    for rank, i in enumerate(order):
        run_ = max(run_, min(1.0, (m - rank) * p[i])); adj[i] = run_
    return adj

SD = np.random.RandomState(99).randint(0, len(SEEDS), (N_BOOT, len(SEEDS)))
_BOOT = {}
def boot_cached(key, y, pred, mask):
    if key not in _BOOT:
        _BOOT[key] = boot_f1(y, pred, mask)
    return _BOOT[key]
def compare(y, pa, pb, mask=None, keys=None):
    sel = slice(None) if mask is None else mask
    per_seed = np.array([f1_score(y[sel], a[sel], average="macro") - f1_score(y[sel], b[sel], average="macro") for a, b in zip(pa, pb)])
    ka, kb = keys
    d = np.stack([boot_cached(ka + (i,), y, a, mask) - boot_cached(kb + (i,), y, b, mask) for i, (a, b) in enumerate(zip(pa, pb))])
    sv = d[SD, np.arange(N_BOOT)[:, None]].mean(1)
    return {"effect_pp": 100 * per_seed.mean(), "per_seed_pp": ", ".join(f"{100 * x:+.2f}" for x in per_seed),
            "lo_pp": 100 * np.percentile(sv, 2.5), "hi_pp": 100 * np.percentile(sv, 97.5), "p": _p(sv)}

def verdict(r):
    if r["p_holm"] < ALPHA and r["effect_pp"] >= MARGIN_PP: return "HELPS"
    if r["p_holm"] < ALPHA and r["effect_pp"] <= -MARGIN_PP: return "HURTS"
    if r["p_holm"] < ALPHA: return "significant, below 1 pp"
    if r["lo_pp"] > -MARGIN_PP and r["hi_pp"] < MARGIN_PP: return "negligible (CI within ±1 pp)"
    return "no detectable effect"

ROWS, PER = [], []
for ds in DATASETS:
    if not all(run_status(ds, r, s)[0] == "done" for r in TEXT_RUNS for s in SEEDS):
        print(f"{ds}: not all 9 runs are finished and verified - skipped in the statistics"); continue
    d, n_cls, _ = load_split(ds); te = d[d.split == "test"]
    y = te.label.values; neg = np.array([any(f_[0] for f_ in eccm_words(t)[1]) for t in te.text])
    P = {r: [np.load(f"{OUT_DIR}/{ds}/{r}_seed{s}_probs.npz")["test"].argmax(1) for s in SEEDS] for r in TEXT_RUNS}
    for r in TEXT_RUNS:
        for s in SEEDS:
            lg = json.load(open(f"{OUT_DIR}/{ds}/{r}_seed{s}_log.json"))["test"]
            PER.append({"dataset": ds, "run": r, "seed": s, **{k: lg[k] for k in ["accuracy", "macro_f1", "ece", "negation_f1"]}})
    for tid, q, a, b in [("T1", "Minimally processed vs conventional cleaning (positive = cleaning hurts)", "R2", "R1"),
                           ("T2", "ECCM cues (no domain anchoring) vs minimally processed", "R3", "R2")]:
        ROWS.append({"family": "primary", "dataset": ds, "id": tid, "question": q, "with": a, "without": b,
                     **compare(y, P[a], P[b], keys=((ds, a, "all"), (ds, b, "all")))})
        ROWS.append({"family": "secondary (negation subset)", "dataset": ds, "id": tid + "-neg", "question": q + ", reviews with negation",
                     "with": a, "without": b, **compare(y, P[a], P[b], neg, keys=((ds, a, "neg"), (ds, b, "neg")))})
if ROWS:
    T = pd.DataFrame(ROWS); T["p_holm"] = np.nan
    for fam in T.family.unique():
        sel = T.family == fam; T.loc[sel, "p_holm"] = holm(T.loc[sel, "p"])
    T["verdict"] = T.apply(verdict, axis=1); T.to_csv(f"{OUT_DIR}/tests.csv", index=False)
    PER = pd.DataFrame(PER); PER.to_csv(f"{OUT_DIR}/results_per_seed.csv", index=False)
    pd.set_option("display.width", 250)
    s = PER.groupby(["dataset", "run"], sort=False)[["accuracy", "macro_f1", "negation_f1", "ece"]].agg(["mean", "std"])
    print("Test results, mean ± SD over 3 seeds (%; ECE as a fraction)")
    print(pd.DataFrame({c: [f"{100 * s.loc[i, (c, 'mean')]:.2f} ± {100 * s.loc[i, (c, 'std')]:.2f}" if c != "ece" else
                            f"{s.loc[i, (c, 'mean')]:.4f} ± {s.loc[i, (c, 'std')]:.4f}" for i in s.index] for c in ["accuracy", "macro_f1", "negation_f1", "ece"]},
                       index=[f"{a} {RUN_NAMES[b]}" for a, b in s.index]).to_string())
    T["effect [95% CI]"] = T.apply(lambda r: f"{r.effect_pp:+.2f} [{r.lo_pp:+.2f}, {r.hi_pp:+.2f}]", axis=1)
    print("\nTESTS (test macro-F1, pp; Holm over 6 within each family). T1 = R2 minimally processed - R1 conventional; "
          "T2 = R3 ECCM cues (no domain anchoring) - R2.")
    print("Bootstrap: individual test reviews are resampled (with seeds). No product or business identifiers exist in these datasets, "
          "so correlated reviews of the same product/business cannot be resampled together; the intervals may be too narrow.")
    print(T[["family", "dataset", "id", "question", "effect [95% CI]", "per_seed_pp", "p_holm", "verdict"]].round(3).to_string(index=False))
    if len(T[T.family == "primary"]) < 6:
        print("\nNOTE: not all datasets are finished, so Holm was applied to fewer than 6 tests. Re-run this cell when all are done.")
    if HASH_FLAGS:
        print(f"\n\033[91mREMINDER: unexpected source-file hashes {HASH_FLAGS} - report them with these results.\033[0m")
    json.dump({"tests": T.drop(columns=["effect [95% CI]"]).to_dict(orient="records"), "unexpected_file_hashes": HASH_FLAGS,
               "fingerprints": {ds: VALID[ds]["fingerprint"] for ds in VALID},
               "bootstrap": "individual test reviews resampled with seeds; no product/business grouping available",
               "rules": {"n_boot": N_BOOT, "margin_pp": MARGIN_PP, "alpha": ALPHA}},
              open(f"{OUT_DIR}/VERDICTS.json", "w"), indent=2, default=str)
    print(f"\nSaved to {OUT_DIR}: tests.csv, results_per_seed.csv, VERDICTS.json (per dataset: data.pkl, validation_report.json, run logs, probabilities)")

Test results, mean ± SD over 3 seeds (%; ECE as a fraction)
                                                accuracy      macro_f1   negation_f1              ece
Flipkart R1 conventional cleaning           88.67 ± 0.27  68.11 ± 1.01  57.77 ± 2.04  0.0286 ± 0.0045
Flipkart R2 minimally processed text        91.79 ± 0.10  72.78 ± 0.48  64.87 ± 1.28  0.0238 ± 0.0002
Flipkart R3 ECCM cues, no domain anchoring  91.95 ± 0.18  73.38 ± 0.86  65.32 ± 1.14  0.0230 ± 0.0021
IMDB R1 conventional cleaning               89.72 ± 0.08  89.72 ± 0.08  89.48 ± 0.08  0.0302 ± 0.0060
IMDB R2 minimally processed text            91.11 ± 0.02  91.11 ± 0.02  90.87 ± 0.04  0.0340 ± 0.0027
IMDB R3 ECCM cues, no domain anchoring      90.99 ± 0.11  90.98 ± 0.11  90.75 ± 0.12  0.0352 ± 0.0029
Yelp R1 conventional cleaning               79.76 ± 0.08  74.46 ± 0.26  74.35 ± 0.27  0.0338 ± 0.0011
Yelp R2 minimally processed text            82.27 ± 0.14  77.93 ± 0.09  77.90 ± 0.10  0.0425 ± 0.0043
Yelp R3 ECCM cues, no 